# Experiment Analysis

This notebook analyzes the experiment results from `20251130` and `20251201`.

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import MaxNLocator
import importlib
import log_parser
from IPython.display import set_matplotlib_formats

# Force reload to pick up latest changes
importlib.reload(log_parser)

# Set SVG rendering
set_matplotlib_formats('svg')

# Set plot style and context
sns.set_theme(style="whitegrid", rc={"grid.linestyle": "--", "grid.alpha": 0.5})
sns.set_context("notebook", font_scale=1)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

In [ ]:
# Define directories
# We check where we are running from to construct paths correctly
cwd = os.getcwd()
print(f"Current working directory: {cwd}")

dirs = ['20251130/base', '20251201/base', '20251202/base', '20251203', '20251204']
# dirs = ['20251203']

if 'workspace/exp/harness' in cwd:
    # Running inside harness dir
    target_dirs = [d for d in dirs]
else:
    # Assuming running from project root
    target_dirs = [os.path.join('workspace/exp/harness', d) for d in dirs]

# Get successful logs
logs = log_parser.get_successful_logs(target_dirs)
print(f"Found {len(logs)} successful logs")

In [ ]:
# Parse logs and extract info
data = []
for log in logs:
    try:
        info = log_parser.get_experiment_info(log)
        score = info.extract_score()
        if score is not None:
            entry = info.params.copy()
            entry['score'] = score
            # Ensure specific columns are present and consistent
            entry['task'] = info.task
            entry['model'] = info.model
            entry['method'] = info.method
            
            # Convert numeric fields
            for k in ['bsz', 'pg', 'graph', 'run', 'topk', 'wind', 'sink']:
                if k in entry:
                    try:
                        if entry[k] != 'auto' and entry[k] != 'none':
                            entry[k] = int(entry[k])
                    except ValueError:
                        pass
            for k in ['frac', 'limit']:
                 if k in entry:
                    try:
                        if entry[k] != 'none':
                            entry[k] = float(entry[k])
                    except ValueError:
                        pass
            data.append(entry)
    except Exception as e:
        print(f"Error processing {log}: {e}")

df = pd.DataFrame(data)
print(f"Parsed {len(df)} records")
if not df.empty:
    display(df.head())
else:
    print("DataFrame is empty. Check logs path or log content.")

In [ ]:
# Pivot table to see results by Task and Method/Model
if not df.empty and 'score' in df.columns:
    pivot = df.pivot_table(index=['task', 'model'], columns=['method'], values='score', aggfunc='mean')
    display(pivot)
else:
    print("Cannot create pivot table: DataFrame is empty or missing 'score' column.")

In [ ]:
# Simple Visualization: Score by Method for each Task
if not df.empty and 'score' in df.columns:
    # Prepare data for plotting
    plot_df = df.copy()
    
    # Formalize Task Names
    task_mapping = {'aime24': 'AIME24', 'longbench': 'LongBench', 'mmlu': 'MMLU', 'ruler': 'RULER'}
    plot_df['task'] = plot_df['task'].map(lambda x: task_mapping.get(x, x))
    
    # Define model colors
    model_colors = {
        'qw_4b': '#1f77b4',      # blue
        'qw_4b_th': '#1f77b4',   # blue
        'qw_30b': '#d62728',     # red
        'qw_30b_th': '#d62728'   # red
    }
    hue_order = ['qw_4b', 'qw_4b_th', 'qw_30b', 'qw_30b_th']
    
    g = sns.catplot(
        data=plot_df,
        x="method", 
        y="score", 
        col="task",
        col_wrap=2, 
        hue="model",
        hue_order=hue_order,
        palette=model_colors,
        kind="bar",
        height=5, 
        aspect=1.2,
        sharey=False,
        legend=False # Custom legend
    )
    for ax in g.axes.flat:
        ax.tick_params(labelbottom=True)
    g.set_axis_labels("", "") # Remove X label
    
    # Bold Titles
    g.set_titles("{col_name}", fontweight='bold')
    
    # Add hatches for _th models
    for ax in g.axes.flat:
        ax.set_ylabel("Score", rotation=0)
        # (x, y) 是相对于坐标轴的比例坐标。
        # x=-0.05 表示在轴左侧一点点，y=1.02 表示在轴顶部上方一点点
        ax.yaxis.set_label_coords(-0.03, 1.02)

        bars = ax.patches
        n_patches = len(bars)
        n_hues = len(hue_order)
        if n_hues > 0 and n_patches % n_hues == 0:
            n_per_hue = n_patches // n_hues
            # th models are at index 1 and 3 in hue_order
            
            # Apply hatch to qw_4b_th
            for i in range(n_per_hue, 2 * n_per_hue):
                bars[i].set_hatch('///')
            
            # Apply hatch to qw_30b_th
            for i in range(3 * n_per_hue, 4 * n_per_hue):
                bars[i].set_hatch('///')
    
    # Create custom legend
    legend_handles = []
    for model in hue_order:
        color = model_colors[model]
        hatch = '///' if '_th' in model else None
        # edgecolor='white' makes the border invisible and the hatch white
        patch = mpatches.Patch(facecolor=color, hatch=hatch, label=model, edgecolor='white')
        legend_handles.append(patch)
    
    # Adjust figure layout to make room for legend
    g.fig.subplots_adjust(top=0.85, hspace=0.2)
    g.fig.legend(handles=legend_handles, loc='lower center', bbox_to_anchor=(0.5, 0.88), ncol=4, frameon=False, fontsize=12)
    
    # Save to PDF
    plt.savefig("assets/score_by_method.pdf", bbox_inches='tight')
    print("Saved score_by_method.pdf")
    
    plt.show()
else:
    print("Cannot plot: DataFrame is empty or missing 'score' column.")


### Quest Analysis: Top-K vs Score

Analysis of how the Top-K parameter affects the score in the `quest` method, comparing different Page Sizes (`pg`) and Models.

In [ ]:
def plot_with_baseline(data_df, x_axis, style_col, title_prefix):
    import matplotlib.patches as mpatches
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    if not data_df.empty:
        plot_data = data_df.copy()

        # Formalize Task Names
        task_mapping = {'aime24': 'AIME24', 'longbench': 'LongBench', 'mmlu': 'MMLU', 'ruler': 'RULER'}
        if 'task' in plot_data.columns:
            plot_data['task'] = plot_data['task'].map(lambda x: task_mapping.get(x, x))

        # Quest plots only use pg=16 results
        if title_prefix == "Quest" and 'pg' in plot_data.columns:
            plot_data = plot_data[plot_data['pg'] == 16]
            if plot_data.empty:
                print("No Quest results found for pg=16.")
                return
        
        # Ensure X-axis is integer (no decimals in ticks)
        if x_axis in plot_data.columns:
            try:
                plot_data[x_axis] = plot_data[x_axis].astype(int)
            except ValueError:
                pass # Keep as is if conversion fails

        # Define model colors and order
        model_colors = {
            'qw_4b': '#1f77b4',      # blue
            'qw_4b_th': '#1f77b4',   # blue
            'qw_30b': '#d62728',     # red
            'qw_30b_th': '#d62728'   # red
        }
        
        available_models = plot_data['model'].unique()
        desired_order = ['qw_4b', 'qw_4b_th', 'qw_30b', 'qw_30b_th']
        hue_order = [m for m in desired_order if m in available_models]

        # Use catplot with kind="bar"
        g = sns.catplot(
            data=plot_data,
            x=x_axis,
            y="score",
            hue="model",
            hue_order=hue_order,
            palette=model_colors,
            col="task",
            col_wrap=2,
            kind="bar",
            height=5,
            aspect=1.2,
            sharey=False,
            sharex=False,
            legend=False
        )

        # Add hatches for _th models
        for ax in g.axes.flat:
            ax.tick_params(labelbottom=True)
            
            bars = ax.patches
            if not bars: continue
            
            num_hues = len(hue_order)
            num_bars = len(bars)
            
            if num_hues > 0:
                # In seaborn barplot, bars are grouped by hue
                bars_per_hue = num_bars // num_hues
                for i, model_name in enumerate(hue_order):
                    if '_th' in model_name:
                        for j in range(i * bars_per_hue, (i + 1) * bars_per_hue):
                            if j < num_bars:
                                bars[j].set_hatch('///')

        # Add Baseline
        # Ensure df is available (global scope in notebook)
        global df
        baseline_df = df[df['method'] == 'base'].copy()
        if not baseline_df.empty:
            # Map tasks in baseline too
            baseline_df['task'] = baseline_df['task'].map(lambda x: task_mapping.get(x, x))
            baseline_scores = baseline_df.groupby(['task', 'model'])['score'].mean().to_dict()

            for ax, task in zip(g.axes.flatten(), g.col_names):
                task_data = plot_data[plot_data['task'] == task]
                models_in_task = task_data['model'].unique()
                
                for model in models_in_task:
                    base_score = baseline_scores.get((task, model))
                    if base_score is not None:
                        color = "#1f77b4" if "4b" in model else "#d62728"
                        linestyle = "--" if "_th" in model else "-"
                        ax.axhline(y=base_score, linestyle=linestyle, color=color, alpha=0.7, linewidth=2)

        # Formalize Axis Labels
        label_map = {
            "topk": "TopK",
            "wind": "Window Size"
        }
        x_label_text = label_map.get(x_axis, x_axis.capitalize())
        g.set_axis_labels(x_label_text, "")
        
        for ax in g.axes.flat:
            ax.set_xlabel(x_label_text)

            ax.set_ylabel("Score", rotation=0)
            # (x, y) 是相对于坐标轴的比例坐标。
            # x=-0.05 表示在轴左侧一点点，y=1.02 表示在轴顶部上方一点点
            ax.yaxis.set_label_coords(-0.03, 1.02)

        # Bold Titles
        g.set_titles("{col_name}", fontweight='bold')

        legend_handles = []
        for model in hue_order:
            color = model_colors[model]
            hatch = '///' if '_th' in model else None
            # edgecolor='white' for white hatch and no visible border
            patch = mpatches.Patch(facecolor=color, hatch=hatch, label=model, edgecolor='white')
            legend_handles.append(patch)

        g.fig.subplots_adjust(top=0.85, hspace=0.25)
        g.fig.legend(handles=legend_handles, loc='lower center', bbox_to_anchor=(0.5, 0.88), ncol=4, frameon=False, fontsize=12)
        
        # Save to PDF
        filename = f"{title_prefix.lower()}_analysis.pdf"
        plt.savefig(filename, bbox_inches='tight')
        print(f"Saved {filename}")

        plt.show()
    else:
        print(f"No data found for {title_prefix}")

if not df.empty and 'score' in df.columns:
    quest_df = df[df['method'] == 'quest'].copy()
    plot_with_baseline(quest_df, "topk", "pg", "Quest")
else:
    print("Cannot plot: DataFrame is empty or missing 'score' column.")

### Stream Analysis: Window Size vs Score

Analysis of how the Window Size (`wind`) parameter affects the score in the `stream` method.

In [ ]:
if not df.empty and 'score' in df.columns:
    # Filter for 'stream' method
    stream_df = df[df['method'] == 'stream'].copy()
    
    # Filter for pg=1 if available, as requested
    # First check if 1 exists in the 'pg' column for stream
    if 'pg' in stream_df.columns:
        # Check values
        available_pgs = stream_df['pg'].unique()
        if 1 in available_pgs:
            print("Filtering Stream results for pg=1 as requested.")
            stream_df = stream_df[stream_df['pg'] == 1]
        else:
            print(f"Note: pg=1 not found in Stream results (available: {available_pgs}). Using all available data.")
    
    plot_with_baseline(stream_df, "wind", "sink", "Stream")
else:
    print("Cannot plot: DataFrame is empty or missing 'score' column.")